In [29]:
import pandas as pd
import psycopg2

pd.set_option('display.max_columns', None)
caminho_do_arquivo = r"..\..\Data\V_OCORRENCIA_AMPLA.json"
df = pd.read_json(caminho_do_arquivo, encoding='utf-8-sig')
df.head(3)

,Numero_da_Ocorrencia,Numero_da_Ficha,Operador_Padronizado,Classificacao_da_Ocorrência,Data_da_Ocorrencia,Hora_da_Ocorrência,Municipio,UF,Regiao,Descricao_do_Tipo,ICAO,Latitude,Longitude,Tipo_de_Aerodromo,Historico,Matricula,Categoria_da_Aeronave,Operador,Tipo_de_Ocorrencia,Fase_da_Operacao,Operacao,Danos_a_Aeronave,Aerodromo_de_Destino,Aerodromo_de_Origem,Lesoes_Fatais_Tripulantes,Lesoes_Fatais_Passageiros,Lesoes_Fatais_Terceiros,Lesoes_Graves_Tripulantes,Lesoes_Graves_Passageiros,Lesoes_Graves_Terceiros,Lesoes_Leves_Tripulantes,Lesoes_Leves_Passageiros,Lesoes_Leves_Terceiros,Ilesos_Tripulantes,Ilesos_Passageiros,Lesoes_Desconhecidas_Tripulantes,Lesoes_Desconhecidas_Passageiros,Lesoes_Desconhecidas_Terceiros,Modelo,CLS,Tipo_ICAO,PMD,Numero_de_Assentos,Nome_do_Fabricante,PSSO
0,7762,201803231711488,ICON TAXI AEREO LTDA E OUTRO,Incidente,2018-03-21,20:00:00,SÃO PAULO,SP,Sudeste,FALHA OU MAU FUNCIONAMENTO DE SISTEMA / COMPON...,NaN,"-23,6261","-46,6564",NaN,DURANTE O VOO O PILOTO SENTIU A AERONAVE COM P...,PRROS,TPX,ICON TAXI AEREO LTDA E OUTRO,SCF-NP,Pouso,Táxi Aéreo,Nenhum,SBSP,SDSC,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,AW109SP,H2T,A109,3175.0,8.0,AGUSTA,verdadeiro
1,7759,201803161337187,EJ ESCOLA DE AERONAUTICA CIVIL LTDA,Acidente,2018-03-14,15:30:00,MONTES CLAROS,MG,Sudeste,COMBUSTÍVEL,Fora de Aeródromo,"-16,7861","-43,8817",-,QUANDO A AERONAVE ATINGIU 6 MILHAS AO SUL DO A...,PREJO,PRI,EJ ESCOLA DE AERONAUTICA LTDA,FUEL,Em rota,Voo de Instrução,Substancial,SBMK,SIMK,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,A152,L1P,C152,757.0,2.0,CESSNA AIRCRAFT,verdadeiro
2,7758,201802271904132,AEROTEX AVIACAO AGRICOLA LTDA.,Acidente,2018-01-26,12:20:00,INACIOLÂNDIA,GO,Centro-Oeste,EXCURSÃO DE PISTA,NaN,"-18,4572","-49,9842",NaN,A AERONAVE DECOLAVA DE UMA PISTA DE POUSO EVEN...,PTWZP,S11,AEROTEX AVIAÇÃO AGRÍCOLA LTDA.,RE,Decolagem,Operação Agrícola,Substancial,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,T188C,L1P,C188,1997.0,1.0,CESSNA AIRCRAFT,verdadeiro


In [31]:
colunas = ["Numero_da_Ocorrencia", "Classificacao_da_Ocorrência", "Data_da_Ocorrencia","Municipio","UF","Regiao","Nome_do_Fabricante"]
df = df[colunas]
df.rename( columns={  'Classificacao_da_Ocorrência' : 'Classificacao_da_Ocorrencia'  } ,inplace=True )

## Método 1 - Insert linha a linha com cursor.execute

In [16]:
# Parâmetros de conexão com o PostgreSQL
dbname   = 'postgres-anac'
user     = 'postgres'
password = '432931'
host     = 'immich-server.tailab6cca.ts.net'
port     = '5432' 

# Abre a conexão e cria o cursor para executar os comandos SQL
conexao = psycopg2.connect(dbname=dbname,
                        user=user,
                        password=password,
                        host=host,
                        port=port)

cursor = conexao.cursor()

# Limpa a tabela antes da carga para evitar duplicidade de histórico
cursor.execute("DELETE FROM public.anac")

# Percorre o DataFrame linha a linha e insere um registro por vez
# Este formato é didático, mas costuma ser mais lento para volumes grandes
for indice,coluna_df in df.iterrows():
    cursor.execute("""   insert into Anac (     
                Numero_da_Ocorrencia, 
                Classificacao_da_Ocorrencia, 
                Data_da_Ocorrencia, 
                Municipio, 
                UF, 
                Regiao, 
                Nome_do_Fabricante
            ) VALUES (%s,%s,%s,%s,%s,%s,%s) 
                
            """ ,(
                coluna_df["Numero_da_Ocorrencia"],
                coluna_df["Classificacao_da_Ocorrencia"],
                coluna_df["Data_da_Ocorrencia"],
                coluna_df["Municipio"],
                coluna_df["UF"],
                coluna_df["Regiao"],
                coluna_df["Nome_do_Fabricante"]                            
            ))

# Grava as alterações no banco
conexao.commit() 

# Fecha os recursos abertos pela conexão
cursor.close()
conexao.close()

## Método 2 - Insert em lote com execute_values

In [ ]:
from psycopg2.extras import execute_values

# Parâmetros de conexão com o PostgreSQL
dbname   = 'postgres-anac'
user     = 'postgres'
password = '432931'
host     = 'immich-server.tailab6cca.ts.net'
port     = '5432' 

# Abre a conexão e prepara o cursor
conexao = psycopg2.connect(dbname=dbname,
                        user=user,
                        password=password,
                        host=host,
                        port=port)

cursor = conexao.cursor()

# Limpa a tabela antes da nova carga
cursor.execute("DELETE FROM public.anac")

# Monta uma lista de tuplas com os dados do DataFrame
# execute_values envia várias linhas em uma única operação
data = [
    (
        row["Numero_da_Ocorrencia"],
        row["Classificacao_da_Ocorrencia"],
        row["Data_da_Ocorrencia"],
        row["Municipio"],
        row["UF"],
        row["Regiao"],
        row["Nome_do_Fabricante"]
    )
    for _, row in df.iterrows()
]

# Define o INSERT em modo batch
insert_query = """
    INSERT INTO Anac (
        Numero_da_Ocorrencia,
        Classificacao_da_Ocorrencia,
        Data_da_Ocorrencia,
        Municipio,
        UF,
        Regiao,
        Nome_do_Fabricante
    ) VALUES %s
"""

# Executa a carga em lote e confirma a transação
execute_values(cursor, insert_query, data)
conexao.commit()

# Fecha cursor e conexão
cursor.close()
conexao.close()

## Método 3 - Carga em massa com COPY

In [18]:
import psycopg2, io

# Parâmetros de conexão com o PostgreSQL
dbname   = 'postgres-anac'
user     = 'postgres'
password = '432931'
host     = 'immich-server.tailab6cca.ts.net'
port     = '5432' 

# Abre a conexão e cria o cursor
conexao = psycopg2.connect(dbname=dbname,
                        user=user,
                        password=password,
                        host=host,
                        port=port)

cursor = conexao.cursor()

# Remove os dados antigos antes da carga em massa
cursor.execute("DELETE FROM public.anac")

# Seleciona apenas as colunas que serão exportadas para o COPY
colunas = ['Numero_da_Ocorrencia', 'Classificacao_da_Ocorrencia',
           'Data_da_Ocorrencia', 'Municipio', 'UF', 'Regiao', 'Nome_do_Fabricante']

# Cria um arquivo CSV na memória para evitar escrita em disco
buffer = io.StringIO()                        # arquivo na RAM
df[colunas].to_csv(buffer, index=False, header=False)
buffer.seek(0)                                # volta ao início

# COPY é a estratégia mais rápida para cargas grandes
# O PostgreSQL lê o CSV diretamente da memória e insere os dados na tabela
cursor.copy_expert(
    sql="""
        COPY public.anac (Numero_da_Ocorrencia,
        Classificacao_da_Ocorrencia,
        Data_da_Ocorrencia,
        Municipio,
        UF,
        Regiao,
        Nome_do_Fabricante)
        FROM STDIN
        WITH (FORMAT CSV, DELIMITER ',', NULL '', QUOTE '"', ENCODING 'UTF8')
    """,
    file=buffer,
)

# Confirma a transação e libera os recursos
conexao.commit()
cursor.close()
conexao.close()

## Método 4 - pandas.to_sql + SQLAlchemy

In [19]:
# pip install sqlalchemy
import pandas as pd
from sqlalchemy import create_engine ,Integer, String, Date,VARCHAR

# Parâmetros de conexão do banco de destino
dbname   = 'postgres-anac'
user     = 'postgres'
password = '432931'
host     = 'immich-server.tailab6cca.ts.net'
port     = '5432'

# Monta a string de conexão usada pelo SQLAlchemy
conexao_str = f'postgresql://{user}:{password}@{host}:{port}/{dbname}'

# Cria a engine para facilitar a integração com o PostgreSQL
engine = create_engine(conexao_str)

nome_tabela = 'anac_sqlalchemy' # Nome da tabela no banco de dados PostgreSQL

# Envia o DataFrame direto para o banco usando to_sql
# UF não cabe em VARCHAR(2) porque o arquivo tem valores como 'Exterior' e 'Indeterminado'
df.to_sql(nome_tabela, engine, index=False, if_exists='replace',dtype={ 
                           'Numero_da_Ocorrencia' :   Integer ,
                           'Classificacao_da_Ocorrencia': VARCHAR(50),
                           'Data_da_Ocorrencia':Date,
                           'Municipio': VARCHAR(100),
                           'UF': VARCHAR(20),
                           'Regiao': VARCHAR(50),
                           'Nome_do_Fabricante': VARCHAR(100)
                           })
# replace = sobrescreve toda a tabela
# append = adicona dados ao fim da tabela 

# Fecha a engine e libera conexões
engine.dispose()

# Velocidade Internet - Teste 01
### 21.6 Mbps transferência
### 4.6 Mbps carregamento

##### Método 01: 11m40s
##### Método 02: 10.1s
##### Método 03: 2.6s
##### Método 04: 6.1s

# Velocidade Internet - Teste 02
### 267.6 Mbps transferência
### 98.0 Mbps carregamento

##### Método 01: 1m22.1s
##### Método 02: 1.6s
##### Método 03: 0.3s
##### Método 04: 1s